In [1]:
import pandapipes as pp
from pandapipes.pf.pipeflow_setup import get_fluid
import pandas as pd
import numpy as np
class ThermoclineTwoLayer():
    def __init__(self, net, name, Flow_control_charge_id,Flow_control_bypass_charge_id,Circ_pump_charge_storage_id,Flow_control_charge_storage_id,Flow_control_discharge_id,
                 Flow_control_bypass_discharge_id,Circ_pump_discharge_storage_id,Flow_control_discharge_storage_id,
                 volume_m3, t_hot_init, t_cold_init, v_hot_fraction_init, UA_loss, UA_interface=0.0):
        self.net = net
        self.fluid = get_fluid(net)
        self.name = name
        self.Flow_control_charge_id = Flow_control_charge_id
        self.Flow_control_bypass_charge_id = Flow_control_bypass_charge_id
        self.Circ_pump_charge_storage_id= Circ_pump_charge_storage_id
        self.Flow_control_charge_storage_id = Flow_control_charge_storage_id
        self.Flow_control_discharge_id = Flow_control_discharge_id
        self.Flow_control_bypass_discharge_id = Flow_control_bypass_discharge_id
        self.Circ_pump_discharge_storage_id = Circ_pump_discharge_storage_id
        self.Flow_control_discharge_storage_id = Flow_control_discharge_storage_id
        self.V_tot = volume_m3
        self.T_hot = t_hot_init
        self.T_cold = t_cold_init
        self.v_hot_fraction = v_hot_fraction_init
        self.UA_loss = UA_loss
        self.UA_interface = UA_interface
        self.V_hot = volume_m3 * self.v_hot_fraction
        self.V_cold = volume_m3 - self.V_hot
        self.V_MIN = volume_m3 * 0.01
        # Inicialización de variables operativas
        self.bypass = False
        self.direction = None
        self.mass_flow = 0.0
        self.mdot_entering = 0.0
        self.mdot_bypass = 0.0
        self.v_in=0
        self.density_average_heat_capacity()

    

    def density_average_heat_capacity(self):
        T_min_K = 273.15 + 20   
        T_max_K = 273.15 + 100  

        # Muestreo fino del rango (cuantos más puntos, más precisa la integral)
        T_samples = np.linspace(T_min_K, T_max_K, 100)

        rho_samples = np.array([self.fluid.get_density(T) for T in T_samples])
        cp_samples = np.array([self.fluid.get_heat_capacity(T) for T in T_samples])

        rho_avg = np.mean(rho_samples)
        cp_avg = np.mean(cp_samples)
        self.density = rho_avg
        self.heat_capacity = cp_avg

    def evaluate_T_network_in(self):
        #Charging 
        if self.mass_flow >= 0:
            self.net.flow_control.at[self.Flow_control_discharge_storage_id, "in_service"] = False  
            self.net.flow_control.at[self.Flow_control_discharge_id, "in_service"] = False
            self.net.flow_control.at[self.Flow_control_bypass_discharge_id, "in_service"] = False
            self.net.flow_control.at[self.Flow_control_bypass_charge_id, "in_service"] = False  
            self.net.flow_control.at[self.Flow_control_charge_storage_id, "in_service"] = True  
            self.net.flow_control.at[self.Flow_control_charge_id, "in_service"] = True
            self.net.circ_pump_mass.at[self.Circ_pump_discharge_storage_id, "mdot_flow_kg_per_s"] = 0
            self.net.circ_pump_mass.at[self.Circ_pump_charge_storage_id, "mdot_flow_kg_per_s"] = self.mass_flow
            self.net.flow_control.at[self.Flow_control_charge_storage_id, "controlled_mdot_kg_per_s"] =self.mass_flow
            self.net.flow_control.at[self.Flow_control_charge_id, "controlled_mdot_kg_per_s"] =self.mass_flow
            self.net.circ_pump_mass.at[self.Circ_pump_charge_storage_id, "t_flow_k"] = self.T_cold
            pp.pipeflow(self.net, mode="bidirectional")
            Tnet=self.net.res_flow_control.at[self.Flow_control_charge_id, "t_from_k"]
        else: #Discharging
            self.net.flow_control.at[self.Flow_control_charge_storage_id, "in_service"] = False  
            self.net.flow_control.at[self.Flow_control_charge_id, "in_service"] = False
            self.net.flow_control.at[self.Flow_control_bypass_charge_id, "in_service"] = False
            self.net.flow_control.at[self.Flow_control_discharge_storage_id, "in_service"] = True  
            self.net.flow_control.at[self.Flow_control_discharge_id, "in_service"] = True
            self.net.flow_control.at[self.Flow_control_bypass_discharge_id, "in_service"] = False
            self.net.circ_pump_mass.at[self.Circ_pump_charge_storage_id, "mdot_flow_kg_per_s"] = 0
            self.net.flow_control.at[self.Flow_control_bypass_charge_id, "controlled_mdot_kg_per_s"] = 0
            self.net.circ_pump_mass.at[self.Circ_pump_discharge_storage_id, "mdot_flow_kg_per_s"] = abs(self.mass_flow)
            self.net.flow_control.at[self.Flow_control_discharge_storage_id, "controlled_mdot_kg_per_s"] = abs(self.mass_flow)
            self.net.flow_control.at[self.Flow_control_discharge_id, "controlled_mdot_kg_per_s"] =abs(self.mass_flow)
            self.net.circ_pump_mass.at[self.Circ_pump_discharge_storage_id, "t_flow_k"] = self.T_hot
            pp.pipeflow(self.net, mode="bidirectional")
            Tnet=self.net.res_flow_control.at[self.Flow_control_discharge_id, "t_from_k"]
        return Tnet
    def evaluate_bypass(self, dt_s,T_net): 
        bypass = False
        mdot_entering = 0.0
        mdot_bypass = 0.0
        v_in=0.0
        if self.mass_flow > 0: # Charge  
            if T_net < self.T_hot:
                self.direction = "to_Tcold"
                mdot_entering=self.mass_flow
            else:
                self.direction = "to_Thot"
                v_available = max(0.0, self.V_cold - self.V_MIN)
                Dv_requested = (self.mass_flow * dt_s) / self.density
                v_in = min(Dv_requested, v_available)
                mdot_entering = (v_in * self.density) / dt_s if dt_s > 0 else 0.0
                mdot_bypass = self.mass_flow - mdot_entering
                if mdot_bypass > 1e-6 or v_available <= 0:
                    bypass = True
                    print(f"  ⚠ [{self.name}] Tanque saturado de calor: "f"{Dv_requested - v_in:.2f} m3 no se pudieron cargar")
                    print(f"  ⚠ [{self.name}] Tanque saturado de calor: "f"{mdot_bypass:.2f} kg/s no se pudieron cargar")
        elif self.mass_flow < 0: # Discharge
            self.direction = "discharge"
            abs_mass_flow = abs(self.mass_flow)
            v_available = max(0.0, self.V_hot - self.V_MIN)
            Dv_requested = (abs_mass_flow * dt_s) / self.density
            v_in = min(Dv_requested, v_available)
            mdot_entering = (v_in * self.density) / dt_s if dt_s > 0 else 0.0
            mdot_bypass = abs_mass_flow - mdot_entering
            if mdot_bypass > 1e-6 or v_available <= 0:
                bypass = True
                print(f"  ⚠ [{self.name}] Tanque saturado de frio: "f"{Dv_requested - v_in:.2f} m3 no se pudieron descargar")
                print(f"  ⚠ [{self.name}] Tanque saturado de frio: "f"{mdot_bypass:.2f} kg/s no se pudieron descargar")
        elif self.mass_flow == 0:
            self.direction = "static"
        self.bypass = bypass
        self.mdot_entering = mdot_entering
        self.mdot_bypass = mdot_bypass
        self.v_in=v_in
    def apply_control_settings(self):
        if self.mass_flow > 0:
            #Deactivate and activate flow controls
            self.net.flow_control.at[self.Flow_control_discharge_storage_id, "in_service"] = False  
            self.net.flow_control.at[self.Flow_control_discharge_id, "in_service"] = False
            self.net.flow_control.at[self.Flow_control_bypass_discharge_id, "in_service"] = False  
            self.net.flow_control.at[self.Flow_control_charge_storage_id, "in_service"] = True  
            self.net.flow_control.at[self.Flow_control_charge_id, "in_service"] = True
            self.net.flow_control.at[self.Flow_control_bypass_charge_id, "in_service"] = True
            self.net.circ_pump_mass.at[self.Circ_pump_discharge_storage_id, "mdot_flow_kg_per_s"] = 0
            self.net.flow_control.at[self.Flow_control_bypass_charge_id, "controlled_mdot_kg_per_s"] = self.mdot_bypass
            self.net.circ_pump_mass.at[self.Circ_pump_charge_storage_id, "mdot_flow_kg_per_s"] = self.mdot_entering
            self.net.flow_control.at[self.Flow_control_charge_storage_id, "controlled_mdot_kg_per_s"] =self.mdot_entering
            self.net.flow_control.at[self.Flow_control_charge_id, "controlled_mdot_kg_per_s"] =self.mass_flow
            self.net.circ_pump_mass.at[self.Circ_pump_charge_storage_id, "t_flow_k"] = self.T_cold
        else: #Discharging 
            self.net.flow_control.at[self.Flow_control_charge_storage_id, "in_service"] = False  
            self.net.flow_control.at[self.Flow_control_charge_id, "in_service"] = False
            self.net.flow_control.at[self.Flow_control_bypass_charge_id, "in_service"] = False
            self.net.flow_control.at[self.Flow_control_discharge_storage_id, "in_service"] = True  
            self.net.flow_control.at[self.Flow_control_discharge_id, "in_service"] = True
            self.net.flow_control.at[self.Flow_control_bypass_discharge_id, "in_service"] = True 
            self.net.circ_pump_mass.at[self.Circ_pump_charge_storage_id, "mdot_flow_kg_per_s"] = 0
            self.net.flow_control.at[self.Flow_control_bypass_charge_id, "controlled_mdot_kg_per_s"] = 0
            self.net.flow_control.at[self.Flow_control_bypass_discharge_id, "controlled_mdot_kg_per_s"] = self.mdot_bypass
            self.net.circ_pump_mass.at[self.Circ_pump_discharge_storage_id, "mdot_flow_kg_per_s"] = self.mdot_entering
            self.net.flow_control.at[self.Flow_control_discharge_storage_id, "controlled_mdot_kg_per_s"] = self.mdot_entering
            self.net.flow_control.at[self.Flow_control_discharge_id, "controlled_mdot_kg_per_s"] =abs(self.mass_flow)
            self.net.circ_pump_mass.at[self.Circ_pump_discharge_storage_id, "t_flow_k"] = self.T_hot

    def Estimate_loss_ambient(self, T_amb):
        self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
        self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
        Q_loss_hot = self.UA_hot_layer * (self.T_hot - T_amb)
        Q_loss_cold = self.UA_cold_layer * (self.T_cold - T_amb)
        return Q_loss_hot, Q_loss_cold
    def V_dis(self, dt_s, T_amb, T_net):
        "The resolution of the equation will be done with euler implicit"
        rho, cp = self.density, self.heat_capacity
        b=(dt_s*self.UA_loss)/(rho*cp*self.V_tot)
        #Euler implicit volume
        if self.mass_flow >= 0:
            if self.direction=='to_Thot':
                self.V_cold -= self.v_in
                self.V_hot += self.v_in
                self.v_hot_fraction = self.V_hot / self.V_tot
                self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
                self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
                a_h = dt_s * self.mdot_entering / (rho * self.V_hot)
                a_c = dt_s * self.mdot_entering / (rho * self.V_cold)
                self.T_hot  = (self.T_hot + a_h * T_net + b * T_amb) / (1 + a_h + b)
                self.T_cold = (self.T_cold + b * T_amb) / (1 + b)
            elif self.direction=='to_Tcold':
                self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
                self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
                a_h = dt_s * self.mdot_entering / (rho * self.V_hot)
                a_c = dt_s * self.mdot_entering / (rho * self.V_cold)
                self.T_cold  = (self.T_cold + a_c * T_net + b * T_amb) / (1 + a_c + b)
                self.T_hot = (self.T_hot + b * T_amb) / (1 + b)
            else:
                self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
                self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
                self.T_hot = (self.T_hot + b * T_amb) / (1 + b)
                self.T_cold = (self.T_cold + b * T_amb) / (1 + b)
        else:
            self.V_cold += self.v_in
            self.V_hot -= self.v_in
            self.UA_hot_layer = self.UA_loss * (self.V_hot / self.V_tot)
            self.UA_cold_layer = self.UA_loss * (self.V_cold / self.V_tot)
            a_h = dt_s * self.mdot_entering / (rho * self.V_hot)
            a_c = dt_s * self.mdot_entering / (rho * self.V_cold)
            self.T_cold  = (self.T_cold + a_c * T_net + b * T_amb) / (1 + a_c + b)
            self.T_hot = (self.T_hot + b * T_amb) / (1 + b)


In [2]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_cons_ida_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_Ida")
j_cons_ida_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_Ida")
j_cons_ida_3=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_Ida")
j_storage_1=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Storage_1")
j_storage_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Storage_2")

#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_cons_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_1_ret")
j_cons_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_2_ret")
j_cons_ret_3=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Cons_3_ret")

#Plant-storage
#Planta Nodo
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15), text_k=273.2, name="Tubo_Ida_Plant_Storage_ida",k_mm=0.1
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_RetornoPlant_Storage_ret",k_mm=0.1)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_Storage_ida_nodo_1",k_mm=0.1
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_Storage_ret_nodo_1",k_mm=0.1
)

#Node Consumers 
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_cons_ida_1,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_1_ida",k_mm=0.1
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_1_ret",k_mm=0.1
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_cons_ida_2,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_2_ida",k_mm=0.1*1000
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_2_ret",k_mm=0.1
)
#Node_Cons3
pipe_ida_5= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_cons_ida_3,
    length_km=0.0441, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_cons_3_ida",k_mm=0.1
)
pipe_retorno_5= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_3, to_junction=j_nodo_2_ret,
    length_km=0.0441, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_3_ret",k_mm=0.1
)
#heat consumer 
HC_1=pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_1 ,
    to_junction=j_cons_ret_1,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_1"
)
#Consm2
HC_2=pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_2,
    to_junction=j_cons_ret_2,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_2"
)
#Consm3
HC_3=pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_3,
    to_junction=j_cons_ret_3,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_3")
#Plant: 
Plant_1=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=360 ,name='Grid'
)
#Charging 
Flow_control_charge=pp.create_flow_control(
    net, from_junction=j_nodo_1, to_junction=j_storage_1,controlled_mdot_kg_per_s=0.1,name="Flow_control_charge"
)
Flow_control_bypass_charge=pp.create_flow_control(
    net, from_junction=j_storage_1, to_junction=j_nodo_1,
    controlled_mdot_kg_per_s=0.1,name="Flow_control_bypass_charge"
)
Circ_pump_charge_storage=pp.create_circ_pump_const_mass_flow(
    net, return_junction=j_storage_1, flow_junction=j_storage_2,
    mdot_flow_kg_per_s=0.1, t_flow_k=300, p_flow_bar=5 ,name="Circ_pump_charge_storage" 
)
Flow_control_charge_storage=pp.create_flow_control(
    net, from_junction=j_storage_2, to_junction=j_nodo_1_ret,
    controlled_mdot_kg_per_s=0.1,name="Flow_control_charge_storage"
)

#discharging 
Flow_control_discharge=pp.create_flow_control(
    net, from_junction=j_nodo_1_ret, to_junction=j_storage_2,
    controlled_mdot_kg_per_s=0.1, name="Flow_control_discharge"
)
Flow_control_bypass_discharge=pp.create_flow_control(
    net, from_junction=j_storage_2, to_junction=j_nodo_1_ret,
    controlled_mdot_kg_per_s=0.1,name="Flow_control_bypass_discharge"
)
Circ_pump_discharge_storage=pp.create_circ_pump_const_mass_flow(
    net, return_junction=j_storage_2, flow_junction=j_storage_1,
    mdot_flow_kg_per_s=0.1, t_flow_k=300, p_flow_bar=5 ,name="Circ_pump_discharge_storage"
)
Flow_control_discharge_storage=pp.create_flow_control(
    net, from_junction=j_storage_1, to_junction=j_nodo_1,
    controlled_mdot_kg_per_s=0.1,name="Flow_control_discharge_storage"
)
Storage=ThermoclineTwoLayer(net, "Storage_1", Flow_control_charge_id=Flow_control_charge, Flow_control_bypass_charge_id=Flow_control_bypass_charge, 
                            Circ_pump_charge_storage_id=Circ_pump_charge_storage, Flow_control_charge_storage_id=Flow_control_charge_storage, Flow_control_bypass_discharge_id=Flow_control_bypass_discharge,
                            Flow_control_discharge_id=Flow_control_discharge,
                              Circ_pump_discharge_storage_id=Circ_pump_discharge_storage, Flow_control_discharge_storage_id=Flow_control_discharge_storage
                              , volume_m3=10, t_hot_init=354, t_cold_init=313, v_hot_fraction_init=2/10, UA_loss=15, UA_interface=0.0)

In [3]:
Data=pd.read_excel("C:\\Users\\sserranose\\OneDrive - INSA Lyon\\Bureau\\Code\\Panda_pipes\\Storage\\Profiles_For_Energy_Storage.xlsx")
Q_consumer_1=Data.iloc[:,2].to_numpy()
Q_consumer_2=Data.iloc[:,3].to_numpy()
Q_consumer_3=Data.iloc[:,4].to_numpy()
Débit_Storage=Data.iloc[:,5].to_numpy()*-1
keys = ["hora","Q_plant_PD", "Q_loss_storage_PD", "Q_Storage_PD", "T_hot_storage_PD", "T_cold_storage_PD", 'V_hot_PD', 'V_cold_PD', 'bypass', 'direction', 'mdot_entering', 'mdot_bypass', 'm_dot_bypass_panda', 'volumen_bypass', 'Tnet_1', "T_net_2"]
results = {key: [] for key in keys}

dt_seconds = 3600
T_ambient = 273.2

for i in range(len(Débit_Storage)):
    print(i+1)
    # 1. Configurar los consumidores de calor
    net.heat_consumer.at[HC_1, "in_service"] = Q_consumer_1[i] > 0
    if Q_consumer_1[i] > 0: net.heat_consumer.at[HC_1, "qext_w"] = Q_consumer_1[i]
    
    net.heat_consumer.at[HC_2, "in_service"] = Q_consumer_2[i] > 0
    if Q_consumer_2[i] > 0: net.heat_consumer.at[HC_2, "qext_w"] = Q_consumer_2[i]
    
    net.heat_consumer.at[HC_3, "in_service"] = Q_consumer_3[i] > 0
    if Q_consumer_3[i] > 0: net.heat_consumer.at[HC_3, "qext_w"] = Q_consumer_3[i]

    mdot_target = Débit_Storage[i]
    Storage.mass_flow = mdot_target
    T_network_preliminar=Storage.evaluate_T_network_in()
    with open("resultados.txt", "a", encoding="utf-8") as f:
        f.write("iteration" +str(i) + "\n")
        f.write("\n\n")
        f.write("TABLA 1 antes\n")
        f.write(Storage.net.flow_control.to_string())
        f.write("\n\n")
        
        f.write("TABLA 2 antes\n")
        f.write(Storage.net.res_flow_control.to_string())
        f.write("\n\n")
        
        f.write("TABLA 3 antes\n")
        f.write(Storage.net.circ_pump_mass.to_string())
        f.write("\n\n")

        f.write("TABLA 4 antes\n")
        f.write(Storage.net.res_circ_pump_mass.to_string())
        f.write("\n\n")

        f.write("TABLA 5 antes\n")
        f.write(Storage.net.res_junction.to_string())
        f.write("\n\n")
    Storage.evaluate_bypass(dt_s=dt_seconds, T_net=T_network_preliminar)
    Storage.apply_control_settings()
    pp.pipeflow(net, mode="bidirectional")
    with open("resultados.txt", "a", encoding="utf-8") as f:
            f.write("iteration" +str(i) + "\n")
            f.write("\n\n")
            f.write("TABLA 1 despues\n")
            f.write(Storage.net.flow_control.to_string())
            f.write("\n\n")
            
            f.write("TABLA 2 despues\n")
            f.write(Storage.net.res_flow_control.to_string())
            f.write("\n\n")
            
            f.write("TABLA 3 despues\n")
            f.write(Storage.net.circ_pump_mass.to_string())
            f.write("\n\n")
    
            f.write("TABLA 4 despues\n")
            f.write(Storage.net.res_circ_pump_mass.to_string())
            f.write("\n\n")
            f.write("TABLA 5 antes\n")
            f.write(Storage.net.res_junction.to_string())
            f.write("\n\n")
    if mdot_target >= 0:
        T_network_definitiva = Storage.net.res_flow_control.at[Storage.Flow_control_charge_id, "t_from_k"]
    else:
        T_network_definitiva = Storage.net.res_flow_control.at[Storage.Flow_control_discharge_id, "t_from_k"]
    Q_loss_hot, Q_loss_cold = Storage.Estimate_loss_ambient(T_amb=T_ambient)
    Storage.V_dis(dt_s=dt_seconds, T_amb=T_ambient,T_net=T_network_definitiva)
    
    # Guardar resultados
    print(f"  ⚠ [{Storage.name}] Temperatura antes: {T_network_preliminar:.2f} K, después: {T_network_definitiva:.2f} K")

    results["Q_plant_PD"].append(float(net.res_circ_pump_pressure.at[Plant_1, "qext_w"]))
    results["hora"].append(i+1)
    results["Q_loss_storage_PD"].append(float(Q_loss_hot + Q_loss_cold))
    results['V_hot_PD'].append(Storage.V_hot)
    results['V_cold_PD'].append(Storage.V_cold)

    if mdot_target >= 0:
        results["Q_Storage_PD"].append(float(net.res_circ_pump_mass.at[Storage.Circ_pump_charge_storage_id, "qext_w"]))
    else:
        results["Q_Storage_PD"].append(float(net.res_circ_pump_mass.at[Storage.Circ_pump_discharge_storage_id, "qext_w"]))

    results["T_hot_storage_PD"].append(float(Storage.T_hot))
    results["T_cold_storage_PD"].append(float(Storage.T_cold))
    results['bypass'].append(Storage.bypass)
    results['direction'].append(Storage.direction)
    results['mdot_entering'].append(Storage.mdot_entering)
    results['mdot_bypass'].append(Storage.mdot_bypass)
    
    if mdot_target >= 0:
        results['m_dot_bypass_panda'].append(float(net.flow_control.at[Storage.Flow_control_bypass_charge_id, "controlled_mdot_kg_per_s"]))
        results['volumen_bypass'].append(float(net.flow_control.at[Storage.Flow_control_bypass_charge_id, "controlled_mdot_kg_per_s"])/Storage.fluid.get_density(T_network_definitiva)*dt_seconds)
    else:
        results['m_dot_bypass_panda'].append(float(net.flow_control.at[Storage.Flow_control_bypass_discharge_id, "controlled_mdot_kg_per_s"]))
        results['volumen_bypass'].append(float(net.flow_control.at[Storage.Flow_control_bypass_discharge_id, "controlled_mdot_kg_per_s"])/Storage.fluid.get_density(T_network_definitiva)*dt_seconds)
        
    results['Tnet_1'].append(float(T_network_preliminar))
    results['T_net_2'].append(T_network_definitiva)


Storage.net.flow_control
Storage.net.res_flow_control
Storage.net.circ_pump_mass
Storage.net.heat_consumer
Storage.V_hot
df=pd.DataFrame(results)
df.to_excel("output_real_thermocline_6_10.xlsx", index=False, sheet_name="Sheet1")


1
  ⚠ [Storage_1] Temperatura antes: 359.79 K, después: nan K
2
  ⚠ [Storage_1] Temperatura antes: 359.85 K, después: 359.85 K
3
  ⚠ [Storage_1] Temperatura antes: 304.77 K, después: 304.77 K
4
  ⚠ [Storage_1] Temperatura antes: 304.73 K, después: 304.73 K
5
  ⚠ [Storage_1] Temperatura antes: 359.82 K, después: 359.82 K
6
  ⚠ [Storage_1] Tanque saturado de calor: 0.10 m3 no se pudieron cargar
  ⚠ [Storage_1] Tanque saturado de calor: 0.03 kg/s no se pudieron cargar
  ⚠ [Storage_1] Temperatura antes: 359.80 K, después: 359.79 K
7
  ⚠ [Storage_1] Tanque saturado de calor: 2.75 m3 no se pudieron cargar
  ⚠ [Storage_1] Tanque saturado de calor: 0.75 kg/s no se pudieron cargar
  ⚠ [Storage_1] Temperatura antes: 359.80 K, después: 359.68 K
8
  ⚠ [Storage_1] Temperatura antes: 304.72 K, después: 304.72 K
9
  ⚠ [Storage_1] Temperatura antes: 304.72 K, después: 304.72 K
10
  ⚠ [Storage_1] Temperatura antes: 359.76 K, después: 359.76 K
11
  ⚠ [Storage_1] Temperatura antes: 304.73 K, después: 304

In [4]:
Storage.net.res_circ_pump_pressure

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.5,3.0,304.143107,360.0,360.0,0.361128,-0.361128,0.000368,-55.856893,84547.001399


In [5]:
Storage.net.res_junction

,p_bar,t_k
0,3.000000,360.000000
1,2.999979,358.101277
2,2.999972,355.831969
3,2.999968,354.414614
4,2.999972,293.150000
5,2.999931,357.643760
6,5.000000,357.575747
7,5.000000,304.568584
8,2.500000,304.143107
9,2.500028,304.568584


In [6]:
Storage.net.res_flow_control

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.500028,5.000000,304.568584,304.568584,304.568584,0.5,-0.5,0.000502
4,5.000000,2.500028,304.568584,304.568584,293.150000,0.0,-0.0,0.000000
5,5.000000,2.999979,357.575747,358.101277,357.575747,0.5,-0.5,0.000516


In [7]:
Storage.net.res_circ_pump_mass

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,5.0,5.0,357.575747,304.568584,306.840057,0.0,-0.0,0.000000,50.735690,-0.000000
1,5.0,5.0,304.568584,357.575747,357.575747,0.5,-0.5,0.000509,-53.007163,111060.146858


In [8]:
Storage.net.res_pipe

,v_mean_m_per_s,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,dp_friction_loss_bar
0,2.111828e-02,3.000000,2.999979,360.000000,358.101277,358.828858,3.611280e-01,-3.611280e-01,3.731908e-04,9.348198e+03,0.024667,2.090063e-05
1,2.053401e-02,2.500028,2.500000,304.568584,304.143107,304.143107,3.611280e-01,-3.611280e-01,3.628659e-04,3.945877e+03,0.034040,2.804465e-05
2,1.057659e-02,2.999979,2.999972,358.101277,355.831969,355.831969,1.811594e-01,-1.811594e-01,1.869037e-04,4.558392e+03,0.031861,6.782441e-06
3,4.896921e-02,2.500143,2.500028,304.748751,304.568584,304.568584,8.611280e-01,-8.611280e-01,8.653573e-04,9.466710e+03,0.024581,1.151645e-04
4,1.056380e-02,2.999972,2.999968,355.831969,354.414614,354.414614,1.811594e-01,-1.811594e-01,1.866777e-04,4.456124e+03,0.032183,4.368206e-06
5,1.030210e-02,2.500150,2.500143,305.000000,304.748751,304.452132,1.811594e-01,-1.811594e-01,1.820532e-04,1.994271e+03,0.049913,6.606824e-06
6,0.000000e+00,2.999972,2.999972,355.831969,293.150000,293.150000,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00,0.449868,0.000000e+00
7,-2.517625e-17,2.500143,2.500143,293.150000,304.748751,293.150000,-4.440892e-16,4.440892e-16,-4.449011e-19,3.774680e-12,0.017821,-1.135019e-35
8,3.972205e-02,2.999979,2.999931,358.101277,357.643760,357.643760,6.799686e-01,-6.799686e-01,7.019466e-04,1.730477e+04,0.021519,4.834958e-05
9,3.867055e-02,2.500201,2.500143,305.000000,304.748751,304.827778,6.799686e-01,-6.799686e-01,6.833650e-04,7.513900e+03,0.026338,5.761074e-05


In [9]:
Storage.density

np.float64(981.4672265252525)

In [10]:
Storage.heat_capacity

np.float64(4189.827381767676)